# WITS Evaluation Pipeline

Quantitative evaluation on Italian WITS dataset with LLM-as-Judge scoring.

**Extension: Semantic Supervision on Italian Wikipedia**

In [1]:
'''!pip uninstall -y tensorflow tensorflow-probability tensorboard protobuf
!pip install -q vllm langchain langchain-community langchain-huggingface rouge_score bert_score datasets matplotlib seaborn
!pip install --upgrade protobuf'''

## 1. Setup

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import warnings
import logging

# Suppress warning messages
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)
import gc
import re
import torch
import json
import spacy
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import VLLM
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

## 2. Configurations

In [3]:
# ArXiv model configuration
SIGEXT_CONFIG = {
    "model_id": "LookUpMark/sigext-wits-it-10k-060t",
    "skip_samples": 25000,
    "threshold": 0.60

}

QUANT_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": torch.float16,
    "bnb_4bit_use_double_quant": True
}
GLOBAL_CONFIG = {
    "llm_model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "judge_model_id": "Qwen/Qwen2.5-14B-Instruct-AWQ",
    "num_test_samples": 100,
    "max_length": 2048,
    "do_sample": False,
    "temperature":0.1,
    "repetition_penalty":1.10,
    "output_dir": "./results_enhanced"
}

GLOBAL_CONFIG["top_p"] = 0.9 if GLOBAL_CONFIG["do_sample"] == True  else 1.0

os.makedirs(GLOBAL_CONFIG["output_dir"], exist_ok=True)

## 3. Enhanced Prompt

In [4]:
# LLM-as-Judge prompt - G_EVAL SOURCE-AWARE Version (Full Source)
GEVAL_PROMPTS = {
    "faithfulness": """<|im_start|>system
You are an expert summary evaluator.
Task: Evaluate the Faithfulness (1-5) of the Summary based on the Source Text.
GOAL: Determine if the summary contains *only* information supported by the source.

Evaluation Steps:
1. Read the summary sentence by sentence.
2. For EACH sentence, search for supporting evidence in the Source. The supporting evidence can be rephrases.
   - IMPORTANT: Look inside headers, lists, and formatted text.
   - IMPORTANT: Accept synonyms (e.g., "capital" matches "capoluogo", "born in" matches dates in brackets).
3. If a claim is not found, verify if it is a logical deduction from the context.
4. If you cannot find a match, explain WHY (e.g. "Masuccio isn't mentioned anywhere in the source")

Criteria:
- 1: Major Hallucination (Contains specific dates, names, or numbers NOT in the source).
- 3: Mostly faithful, but includes some minor unverifiable details or exaggerations.
- 5: Perfectly Faithful (Every piece of information is supported by the source text, headers, or lists).

Output format:
- Analysis: [Map each summary sentence to a source snippet].
- Verdict: [Explain the score].
- Final: "Score: X"
<|im_end|>
<|im_start|>user
Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>
<|im_end|>
<|im_start|>assistant
""",

    "completeness": """<|im_start|>system
You are an expert Content Analyst.
Task: Evaluate the Completeness (1-5) of the Summary.
GOAL: Determine if the summary captures the MAIN EVENT/TOPIC of the source, ignoring minor details.

Evaluation Steps:
1. Analyze the Source Text and identify the most critical "Who/Where/When/Why".
   - Ignore metadata, file names, or minor side notes.
2. Check if these SPECIFIC core facts are present in the Summary.
   - Allow for rephrasing (e.g., if source says "died in 1990", summary saying "passed away in the 90s" is acceptable coverage).
3. Determine if the summary is complete or leaves the reader confused.

Criteria:
- 1: Irrelevant. Misses the main topic completely.
- 3: Partial. Mentions the topic but misses a crucial fact (e.g., who did it, or the main result).
- 5: Comprehensive. Covers all core entities and main events described in the source.

Output format:
- Key factor identified: [List the Key Facts found in Source].
- Presence in the summary: [YES/NO].
- Final: "Score: X"
<|im_end|>
<|im_start|>user
Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>
<|im_end|>
<|im_start|>assistant
""",

    "conciseness": """<|im_start|>system
You are an expert Editor.
Task: Evaluate the Conciseness (1-5) of the Summary.
GOAL: Determine if the summary is efficient and dense, or verbose and repetitive.

Evaluation Steps:
1. Check for repetitive phrases or redundant adjectives.
2. Check if the sentence structure is unnecessarily complex.
3. Verify if the summary packs a lot of information into few words (High Density).

Criteria:
- 1: Verbose/Repetitive. Uses 20 words where 5 would do. Repeats the same information.
- 3: Average. Readable but contains some filler words or slight redundancy.
- 5: Highly Concise. Every word serves a purpose. High information density.

Output format:
- Analysis: [Brief comment on style].
- Final: "Score: X"
<|im_end|>
<|im_start|>user
Input Data:
<summary_text>
{generated}
</summary_text>
<|im_end|>
<|im_start|>assistant
""",

    "abstraction": """<|im_start|>system
You are an expert Linguist.
Task: Evaluate the Abstraction (1-5) of the Summary.
GOAL: Determine if the model wrote a new text (Abstractive) or just copied sentences (Extractive).

Evaluation Steps:
1. Compare the vocabulary and sentence structure of the Source and Summary.
2. Look for "n-gram overlap". Are there long sequences of words identical to the source?
3. Did the model synthesize information (combine two source sentences into one summary sentence)?

Criteria:
- 1: Copy-Paste. The summary copies long phrases (10+ words) taken from the source.
- 3: Mixed. Some rephrasing, but relies heavily on original phrasing/cliches.
- 5: Highly Abstractive. The summary uses entirely new vocabulary and sentence structures to convey the same meaning.

Output format:
- Analysis: [Check for copied phrases]
- Final: "Score: X"
<|im_end|>
<|im_start|>user
Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>
<|im_end|>
<|im_start|>assistant
"""
}

## 4. Functions

In [5]:
def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

clear_gpu_memory()

## 5. New Abstraction Metrics

In [6]:
def get_ngrams(text, n=3):
    """Extract n-grams from text."""
    words = text.lower().split()
    return [tuple(words[i:i+n]) for i in range(len(words)-n+1)]


def compute_abstraction_score(source, generated, n=3):
    """
    Compute abstraction score: 1 - (n-gram overlap with source).
    Higher = more abstractive (less copying).
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)

    if not gen_ngrams:
        return 1.0  # Empty summary = no copying

    copied = sum(1 for ng in gen_ngrams if ng in source_ngrams)
    copy_ratio = copied / len(gen_ngrams)

    return 1.0 - copy_ratio


def compute_compression_ratio(source, generated):
    """Compute compression ratio (lower = more compressed)."""
    if len(source) == 0:
        return 1.0
    return len(generated) / len(source)


def compute_novel_ngrams(source, generated, n=2):
    """
    Compute percentage of n-grams in generated that are NOT in source.
    Higher = more novel content.
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)

    if not gen_ngrams:
        return 0.0

    novel = sum(1 for ng in gen_ngrams if ng not in source_ngrams)
    return novel / len(gen_ngrams)

## 6. LLM-as-Judge Evaluation

In [7]:
def parse_judge_response(text):
    """Robust JSON parsing for LLM judge responses."""
    import re
    import json

    json_match = re.search(r'Score:\s*(\d)', text, re.IGNORECASE)

    if json_match:
        score = int(json_match.group(1))
        return max(1, min(5, score))
    return 3


def llm_judge_evaluate(source, generated, judge_chain):
    """Use LLM to judge quality of generated summary against source."""
    scores = {}

    for metric_name, chain in judge_chain.items():
        try:
          inputs = {"generated": generated} # generated for conciseness
          if "{source}" in GEVAL_PROMPTS[metric_name]:
                inputs["source"] = source   # source for faithfulness, completeness and abstraction
          result = chain.invoke(inputs)

          result_text = result.strip()

          score = parse_judge_response(result_text)
          scores[metric_name] = score

          scores[f"{metric_name}_reason"] = result_text[:200] + "..." # Save 200 characters

        except Exception as e:
            print(f"      Judge error on {metric_name}: {e}")
            scores[metric_name] = 3 # Default
            scores[f"{metric_name}_reason"] = str(e)

In [8]:
def run_enhanced_evaluation(generated_data, judge_chain):
    """Run evaluation with traditional + abstraction + LLM-judge metrics."""

    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

    metrics = {
        # Traditional
        "bert": [], "rouge1": [], "kir": [],
        # Abstraction
        "abstraction": [], "compression": [], "novel_ngrams": [],
        # LLM Judge
        "judge_faithfulness": [], "judge_completeness": [],
        "judge_conciseness": [], "judge_abstraction": []
    }
    samples = []

    for item in tqdm(generated_data, desc="    Evaluating"):
          try:
            gen_summary = item['generated_summary']
            salient_sents = item['salient_sentences']
            source = item['source']
            # === TRADITIONAL METRICS ===

            # ROUGE-1 and ROUGE-L
            rouge_scores = scorer.score(item['reference'], gen_summary)
            metrics["rouge1"].append(rouge_scores['rouge1'].fmeasure)

            # BERTScore (Italian)
            _, _, F1 = bert_score([gen_summary], [item['reference']], lang="it", verbose=False)
            bert_sc = F1.mean().item()
            metrics["bert"].append(bert_sc)

            # KIR
            kir_score = 0.0
            if salient_sents:
                gen_lower = gen_summary.lower()
                hits = 0
                for sent in salient_sents:
                    words = [w.lower() for w in sent.split() if len(w) > 4]
                    if words:
                        word_hits = sum(1 for w in words if w in gen_lower)
                        if word_hits / len(words) > 0.3:
                            hits += 1
                kir_score = hits / len(salient_sents)
            metrics["kir"].append(kir_score)

            # === ABSTRACTION METRICS ===

            abstraction = compute_abstraction_score(item['source'], gen_summary)
            compression = compute_compression_ratio(item['source'], gen_summary)
            novel = compute_novel_ngrams(item['source'], gen_summary)

            metrics["abstraction"].append(abstraction)
            metrics["compression"].append(compression)
            metrics["novel_ngrams"].append(novel)

            # === LLM JUDGE ===

            judge_scores = llm_judge_evaluate(source, gen_summary, judge_chain)

            for key, val in judge_scores.items():
                if key in metrics:
                    metrics[key].append(val)

            # Store sample details
            samples.append({
                "source": item['source'][:500] + "..." if len(item['source']) > 500 else item['source'],
                "reference": item['reference'],
                "salient_sentences": salient_sents,
                "generated_summary": gen_summary,
                "scores": {
                    "bert": float(bert_sc),
                    "rouge1": float(rouge_scores['rouge1'].fmeasure),
                    "kir": float(kir_score),
                    "abstraction": float(abstraction),
                    "compression": float(compression),
                    "novel_ngrams": float(novel),
                    "judge": judge_scores
                }
            })

          except Exception as e:
            print(f"    Error: {e}")
            continue

    return metrics, samples

## 7. Main Evaluation Loop

In [9]:
input_file = "generation_output.json"

In [10]:
with open(input_file, "r") as f:
    data = json.load(f)
print(f"Loaded {len(data)} samples for the validation")

In [11]:
vllm_engine_args = {
    "max_num_seqs": 8,
    "gpu_memory_utilization": 0.7,
    "enforce_eager": True,
    "disable_log_stats": True
}

llm_judge = VLLM(
    model=GLOBAL_CONFIG["judge_model_id"],
    quantization="awq",
    dtype="float16",
    max_model_len=4096,
    trust_remote_code=True,
    vllm_kwargs=vllm_engine_args
)
llm_judge.temperature = 0.0
llm_judge.max_new_tokens = 512

# Create chains
judge_chains = {}
stop_tokens = ["<|im_end|>"]
binder = llm_judge.bind(stop=stop_tokens)

for metric, prompt_txt in GEVAL_PROMPTS.items():
    input_vars = ['generated']
    if '{source}' in prompt_txt: input_vars.append('source')
    prompt = PromptTemplate(template=prompt_txt, input_variables=input_vars)
    judge_chains[metric] = prompt | binder | StrOutputParser()

metrics, samples = run_enhanced_evaluation(data, judge_chains)

In [12]:
# Compute and display results
run_info = {
    "timestamp": datetime.now().isoformat(),
    "sigext_model": SIGEXT_CONFIG["model_id"],
    "llm_model": GLOBAL_CONFIG["llm_model_id"],
    "num_samples": len(samples),
    "prompt_type": "optimized_abstractive_v3"
}

if GLOBAL_CONFIG["do_sample"]:
   if "temperature" in GLOBAL_CONFIG:
       run_info["temperature"] = GLOBAL_CONFIG["temperature"]
   if "top_p" in GLOBAL_CONFIG:
       run_info["top_p"] = GLOBAL_CONFIG["top_p"]
   if "repetition_penalty" in GLOBAL_CONFIG:
       run_info["repetition_penalty"] = GLOBAL_CONFIG["repetition_penalty"]
   if 'adapter' in GLOBAL_CONFIG:
       run_info["adapter"] = GLOBAL_CONFIG['adapter']

results = {
    "run_info": run_info,
    "metrics": {}
}

# Aggregate metrics
for m in ["bert", "rouge1", "kir", "abstraction", "compression", "novel_ngrams"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

for m in ["judge_faithfulness", "judge_completeness", "judge_conciseness", "judge_abstraction"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

if metrics.get("judge_faithfulness"):
    overall = [sum(metrics[f"judge_{k}"][i] for k in ["faithfulness", "completeness", "conciseness", "abstraction"])
               for i in range(len(metrics["judge_faithfulness"]))]
    results["metrics"]["judge_overall"] = {"mean": float(np.mean(overall)) / 4, "std": float(np.std(overall)) / 4}

results["samples"] = samples

# Print summary
print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

m = results["metrics"]
if "bert" in m:
    print(f"\nTraditional Metrics:")
    print(f"  BERT Score:  {m['bert']['mean']:.4f} +/- {m['bert']['std']:.4f}")
    print(f"  ROUGE-1:     {m['rouge1']['mean']:.4f} +/- {m['rouge1']['std']:.4f}")
    print(f"  KIR:         {m['kir']['mean']:.2%}")

if "abstraction" in m:
    print(f"\nAbstraction Metrics:")
    print(f"  Abstraction: {m['abstraction']['mean']:.4f}")
    print(f"  Novel:       {m['novel_ngrams']['mean']:.2%}")
    print(f"  Compression: {m['compression']['mean']:.2%}")

if "judge_faithfulness" in m:
    print(f"\nLLM Judge (1-5):")
    print(f"  Faithfulness: {m['judge_faithfulness']['mean']:.2f}")
    print(f"  Completeness: {m['judge_completeness']['mean']:.2f}")
    print(f"  Conciseness:  {m['judge_conciseness']['mean']:.2f}")
    print(f"  Abstraction:  {m['judge_abstraction']['mean']:.2f}")
    print(f"  Overall:      {m['judge_overall']['mean']:.2f}")

# Save
output_file = os.path.join(GLOBAL_CONFIG["output_dir"], "results_enhanced.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {output_file}")


In [13]:
# Sample outputs
print("=" * 60)
print("SAMPLE OUTPUTS")
print("=" * 60)

for idx in [0, len(samples)//2, len(samples)-1]:
    s = samples[idx]
    print(f"\n[Sample {idx}]")
    print(f"Reference: {s['reference'][:200]}...")
    print(f"Generated: {s['generated_summary'][:200]}...")
    sc = s['scores']
    print(f"Scores: BERT={sc['bert']:.2f} ROUGE={sc['rouge1']:.2f} KIR={sc['kir']:.2f} Abstr={sc['abstraction']:.2f}")


## 8. Cleanup

In [14]:
# Cleanup
del llm_judge, judge_chains
clear_gpu_memory()

print(" Cleanup complete!")